# Agent智能体
参考文档：https://hello-agents.datawhale.cc/

在人工智能领域，智能体被定义为任何能够通过传感器（Sensors）感知其所处环境（Environment），并自主地通过执行器（Actuators）采取行动（Action）以达成特定目标的实体。

## 实现最简单的智能体

```mermaid
flowchart LR
    subgraph Agent
        direction TB
        a2(LLM) <--> a4(Tool)
        a2(LLM) <--> a5(Memory)
    end
    a1(input) --> Agent
    Agent --> a3(output)

### 安装所需的库

In [1]:
#!pip install requests openai
!pip show requests openai

Name: requests
Version: 2.34.2
Summary: Python HTTP for Humans.
Home-page: 
Author: 
Author-email: Kenneth Reitz <me@kennethreitz.org>
License: Apache-2.0
Location: /home/kokomi/anaconda3/envs/mamba/lib/python3.12/site-packages
Requires: certifi, charset_normalizer, idna, urllib3
Required-by: datasets, flashinfer-python, gguf, jupyterlab_server, mistral_common, modelscope, opentelemetry-exporter-otlp-proto-http, ray, tavily-python, tiktoken, vllm
---
Name: openai
Version: 2.37.0
Summary: The official Python library for the openai API
Home-page: https://github.com/openai/openai-python
Author: 
Author-email: OpenAI <support@openai.com>
License: Apache-2.0
Location: /home/kokomi/anaconda3/envs/mamba/lib/python3.12/site-packages
Requires: anyio, distro, httpx, jiter, pydantic, sniffio, tqdm, typing-extensions
Required-by: vllm


tavily-python是一个强大的 AI 搜索 API 客户端，用于获取实时的网络搜索结果，可以在[官网](https://app.tavily.com/home)注册后获取 API

In [2]:
# !pip install tavily-python 
!pip show tavily-python

Name: tavily-python
Version: 0.7.25
Summary: Python wrapper for the Tavily API
Home-page: https://github.com/tavily-ai/tavily-python
Author: Tavily AI
Author-email: support@tavily.com
License: 
Location: /home/kokomi/anaconda3/envs/mamba/lib/python3.12/site-packages
Requires: httpx, requests, tiktoken
Required-by: 


In [3]:
# 设置你的apikey
%env TAVILY_API_KEY=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

env: TAVILY_API_KEY=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


### 系统指令模版

In [36]:
AGENT_SYSTEM_PROMPT = """
你是一个智能旅行助手。你的任务是分析用户的请求，并使用可用工具一步步地解决问题。

# 可用工具:
- `get_weather(city: str)`: 查询指定城市的实时天气。
- `get_attraction(city: str, weather: str)`: 根据城市和天气搜索推荐的旅游景点。

# 输出格式要求:
你的每次回复必须严格遵循以下格式，包含一对Thought和Action：

Thought: [你的思考过程和下一步计划]
Action: [你要执行的具体行动]

Action的格式必须是以下之一！：
1. 调用工具：function_name(arg_name="arg_value")，例如：get_weather(city="北京") 。
2. 结束任务：Finish[你的最终答案内容]。

# 重要提示:
- 每次只输出一对Thought-Action
- Action必须在同一行，不要换行
- 当收集到足够信息可以回答用户问题时，必须使用 Action: Finish[你的最终答案内容] 格式结束

请开始吧！
"""

### 工具1：查询真实天气
使用免费天气api服务 wttr.in

In [5]:
import requests

def get_weather(city: str) -> str:
    """
    通过调用 wttr.in API 查询今天真实的天气信息。
    """
    print("正在调用tool:get_weathern......")
    # API端点，我们请求JSON格式的数据
    url = f"https://wttr.in/{city}?format=j1"
    
    try:
        # 发起网络请求
        response = requests.get(url)
        # 检查响应状态码是否为200 (成功)
        response.raise_for_status() 
        # 解析返回的JSON数据
        data = response.json()
            
        # 提取当前天气状况
        current_condition = data['current_condition'][0]
        weather_desc = current_condition['weatherDesc'][0]['value']
        temp_c = current_condition['temp_C']
        # print("今天天气内容：",current_condition)
        
        # 格式化成自然语言返回
        return f"{city}当前天气:{weather_desc}，气温{temp_c}摄氏度"
        
    except requests.exceptions.RequestException as e:
        # 处理网络错误
        return f"错误:查询天气时遇到网络问题 - {e}"
    except (KeyError, IndexError) as e:
        # 处理数据解析错误
        return f"错误:解析天气数据失败，可能是城市名称无效 - {e}"

In [6]:
get_weather("苏州")

正在调用tool:get_weathern......


'苏州当前天气:Partly Cloudy ，气温32摄氏度'

### 工具2：搜索并推荐旅游景点

In [7]:
import os
from tavily import TavilyClient

def get_attraction(city: str, weather: str) -> str:
    """
    根据城市和天气，使用Tavily Search API搜索并返回优化后的景点推荐。
    """
    # 1. 从环境变量中读取API密钥
    print("正在调用tool:get_attraction......")
    api_key = os.environ.get("TAVILY_API_KEY")
    if not api_key:
        return "错误:未配置TAVILY_API_KEY环境变量。"

    # 2. 初始化Tavily客户端
    tavily = TavilyClient(api_key=api_key)
    
    # 3. 构造一个精确的查询
    query = f"'{city}' 在'{weather}'天气下最值得去的旅游景点推荐及理由"
    
    try:
        # 4. 调用API，include_answer=True会返回一个综合性的回答
        response = tavily.search(query=query, search_depth="basic", include_answer=True)
        
        # 5. Tavily返回的结果已经非常干净，可以直接使用
        # response['answer'] 是一个基于所有搜索结果的总结性回答
        if response.get("answer"):
            return response["answer"]
        
        # 如果没有综合性回答，则格式化原始结果
        formatted_results = []
        for result in response.get("results", []):
            formatted_results.append(f"- {result['title']}: {result['content']}")
        
        if not formatted_results:
             return "抱歉，没有找到相关的旅游景点推荐。"

        return "根据搜索，为您找到以下信息:\n" + "\n".join(formatted_results)

    except Exception as e:
        return f"错误:执行Tavily搜索时出现问题 - {e}"

In [9]:
get_attraction("苏州", get_weather("苏州"))

正在调用tool:get_weathern......
正在调用tool:get_attraction......


"Under partly cloudy skies with a temperature of 32°C, visit the Humble Administrator's Garden for a serene experience. This historic garden offers a peaceful retreat and showcases classical Chinese garden design."

### 将工具都放在工具箱里

In [10]:
# 将所有工具函数放入一个字典，方便后续调用
available_tools = {
    "get_weather": get_weather,
    "get_attraction": get_attraction,
}

### 使用LLM推理

In [136]:
import torch
import re
import gc
from modelscope import AutoModelForCausalLM, AutoTokenizer

def CleanMemory():
    torch.cuda.empty_cache()
    gc.collect()

#### 使用Qwen/Qwen3-0.6B本地模型推理，在6_LMandLora.ipynb里已经下载过了

In [12]:
!modelscope download --model Qwen/Qwen3-0.6B --local_dir ./Qwen/Qwen3-0.6B


 _   .-')                _ .-') _     ('-.             .-')                              _ (`-.    ('-.
( '.( OO )_             ( (  OO) )  _(  OO)           ( OO ).                           ( (OO  ) _(  OO)
 ,--.   ,--.).-'),-----. \     .'_ (,------.,--.     (_)---\_)   .-----.  .-'),-----.  _.`     \(,------.
 |   `.'   |( OO'  .-.  ',`'--..._) |  .---'|  |.-') /    _ |   '  .--./ ( OO'  .-.  '(__...--'' |  .---'
 |         |/   |  | |  ||  |  \  ' |  |    |  | OO )\  :` `.   |  |('-. /   |  | |  | |  /  | | |  |
 |  |'.'|  |\_) |  |\|  ||  |   ' |(|  '--. |  |`-' | '..`''.) /_) |OO  )\_) |  |\|  | |  |_.' |(|  '--.
 |  |   |  |  \ |  | |  ||  |   / : |  .--'(|  '---.'.-._)   \ ||  |`-'|   \ |  | |  | |  .___.' |  .--'
 |  |   |  |   `'  '-'  '|  '--'  / |  `---.|      | \       /(_'  '--'\    `'  '-'  ' |  |      |  `---.
 `--'   `--'     `-----' `-------'  `------'`------'  `-----'    `-----'      `-----'  `--'      `------'


Successfully Downloaded from model Qwen/Qwen3-0.6B.


In [54]:
def load_model(model_name="./Qwen/Qwen3-0.6B"):
    # 加载分词器
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    # 加载模型
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype=torch.bfloat16,
        # dtype=torch.float16, #上面的不能用就用下面的
        device_map="auto"
    )
    return model, tokenizer

In [14]:
model, tokenizer = load_model("./Qwen/Qwen3-0.6B")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

#### 输入提示词

In [43]:
user_prompt = "你好，请帮我查询一下今天苏州的天气，然后根据天气推荐一个合适的旅游景点。"
prompt_history = [f"用户请求: {user_prompt}"]

print(f"用户输入: {user_prompt}")

用户输入: 你好，请帮我查询一下今天苏州的天气，然后根据天气推荐一个合适的旅游景点。


In [44]:
def get_qwen_output(messages, model, tokenizer):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True # 是否使用思考模式
    )
    # 编码
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    # 推理
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=32768 #最大上下文长度
    )
    # 得到输出并转 ids
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
    try:
        # 找到思考结束的符号的id的位置 151668 (</think>)
        index = len(output_ids) - output_ids[::-1].index(151668)
    except ValueError:
        index = 0
    # 思考内容
    thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
    # 输出内容
    content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")
    return thinking_content, content

### Agent行动循环

In [45]:
for i in range(5): # 设置最大循环次数
    print(f"--- 循环 {i+1} ---\n")
    
    # 3.1. 构建Prompt
    full_prompt = "\n".join(prompt_history)
    # 封装到messages
    messages = [
        {'role': 'system', 'content': AGENT_SYSTEM_PROMPT},
        {'role': 'user', 'content': full_prompt}
    ]
    print("历史记录", messages)
    # 3.2. 调用LLM进行思考
    think_output, output = get_qwen_output(messages, model, tokenizer)
    print(f"模型think:\n{think_output}\n")
    # 模型可能会输出多余的Thought-Action，需要截断
    match = re.search(r'(Thought:.*?Action:.*?)(?=\n\s*(?:Thought:|Action:|Observation:)|\Z)', output, re.DOTALL)
    if match:
        truncated = match.group(1).strip()
        if truncated != output.strip():
            output = truncated
            print("已截断多余的 Thought-Action 对")
    print(f"模型输出:\n{output}\n")
    prompt_history.append(output)
    
    # 3.3. 解析并执行行动
    action_match = re.search(r"Action: (.*)", output, re.DOTALL)
    if not action_match:
        observation = "错误: 未能解析到 Action 字段。请确保你的回复严格遵循 'Thought: ... Action: ...' 的格式。"
        observation_str = f"Observation: {observation}"
        print(f"{observation_str}\n" + "="*40)
        prompt_history.append(observation_str)
        continue
    action_str = action_match.group(1).strip()

    # if action_str.startswith("Finish"):
    if "Finish" in action_str:
        final_answer = re.search(r"Finish\[(.*)\]", action_str).group(1)
        print(f"任务完成，最终答案: {final_answer}")
        break
    
    tool_name = re.search(r"(\w+)\(", action_str).group(1)
    args_str = re.search(r"\((.*)\)", action_str).group(1)
    kwargs = dict(re.findall(r'(\w+)="([^"]*)"', args_str))

    if tool_name in available_tools:
        observation = available_tools[tool_name](**kwargs)
    else:
        observation = f"错误:未定义的工具 '{tool_name}'"

    # 3.4. 记录观察结果
    observation_str = f"Observation: {observation}"
    print(f"{observation_str}\n" + "="*40)
    prompt_history.append(observation_str)

--- 循环 1 ---

历史记录 [{'role': 'system', 'content': '\n你是一个智能旅行助手。你的任务是分析用户的请求，并使用可用工具一步步地解决问题。\n\n# 可用工具:\n- `get_weather(city: str)`: 查询指定城市的实时天气。\n- `get_attraction(city: str, weather: str)`: 根据城市和天气搜索推荐的旅游景点。\n\n# 输出格式要求:\n你的每次回复必须严格遵循以下格式，包含一对Thought和Action：\n\nThought: [你的思考过程和下一步计划]\nAction: [你要执行的具体行动]\n\nAction的格式必须是以下之一！：\n1. 调用工具：function_name(arg_name="arg_value")，例如：get_weather(city="北京") 。\n2. 结束任务：Finish[你的最终答案内容]。\n\n# 重要提示:\n- 每次只输出一对Thought-Action\n- Action必须在同一行，不要换行\n- 当收集到足够信息可以回答用户问题时，必须使用 Action: Finish[你的最终答案内容] 格式结束\n\n请开始吧！\n'}, {'role': 'user', 'content': '用户请求: 你好，请帮我查询一下今天苏州的天气，然后根据天气推荐一个合适的旅游景点。'}]
模型think:
<think>
好的，用户让我查询苏州今天的天气，然后根据天气推荐旅游景点。首先，我需要确认用户的需求。用户可能是在计划去苏州的旅行，所以需要知道天气情况来安排行程。接下来，我应该使用get_weather工具来获取苏州的实时天气数据。然后，使用get_attraction工具，传入苏州和查询到的天气信息，得到推荐的景点。

首先，我需要调用get_weather函数，参数是city="苏州"。然后，得到天气数据后，再调用get_attraction，传入city="苏州"和weather参数。在调用这两个工具后，应该能获取到具体的天气情况和景点推荐，最后将结果整理成用户友好的回答。
</think>

已截断多余的 Thought-Action 对
模型输出:
Thought: 调用get_weathe

In [84]:
del model
del tokenizer
CleanMemory()

**如果结果有问题，那就对了，下面是使用LLM正确输出样例**

```
用户输入: 你好，请帮我查询一下今天北京的天气，然后根据天气推荐一个合适的旅游景点。
========================================
--- 循环 1 ---

正在调用大语言模型...
大语言模型响应成功。
模型输出:
Thought: 首先需要获取北京今天的天气情况，之后再根据天气情况来推荐旅游景点。
Action: get_weather(city="北京")

Observation: 北京当前天气:Sunny，气温26摄氏度
========================================      
--- 循环 2 ---

正在调用大语言模型...
大语言模型响应成功。
模型输出:
Thought: 现在已经知道了北京今天的天气是晴朗且温度适中，接下来可以基于这个信息来推荐一个适合的旅游景点了。
Action: get_attraction(city="北京", weather="Sunny")

Observation: 北京在晴天最值得去的旅游景点是颐和园，因其美丽的湖景和古建筑。另一个推荐是长城，因其壮观的景观和历史意义。
========================================
--- 循环 3 ---

正在调用大语言模型...
大语言模型响应成功。
模型输出:
Thought: 已经获得了两个适合晴天游览的景点建议，现在可以根据这些信息给用户提供满意的答复。
Action: Finish[今天北京的天气是晴朗的，气温26摄氏度，非常适合外出游玩。我推荐您去颐和园欣赏美丽的湖景和古建筑，或者前往长城体验其壮观的景观和深厚的历史意义。希望您有一个愉快的旅行！]

任务完成，最终答案: 今天北京的天气是晴朗的，气温26摄氏度，非常适合外出游玩。我推荐您去颐和园欣赏美丽的湖景和古建筑，或者前往长城体验其壮观的景观和深厚的历史意义。希望您有一个愉快的旅行！
```

## ReAct

ReAct范式通过一种特殊的提示工程来引导模型，使其每一步的输出都遵循一个固定的轨迹：

- Thought (思考)： 这是智能体的“内心独白”。它会分析当前情况、分解任务、制定下一步计划，或者反思上一步的结果。
- Action (行动)： 这是智能体决定采取的具体动作，通常是调用一个外部工具，例如 Search['华为最新款手机']。
- Observation (观察)： 这是执行Action后从外部工具返回的结果，例如搜索结果的摘要或API的返回值。

智能体将不断重复这个 Thought -> Action -> Observation 的循环，将新的观察结果追加到历史记录中，形成一个不断增长的上下文，直到它在Thought中认为已经找到了最终答案，然后输出结果。

```mermaid
flowchart LR
    a(Tools) <--Function Call--> b(LLM) --Action-->c(Env)
    c --Observation-->b

### 系统提示词

In [128]:
# ReAct 提示词模板
REACT_PROMPT_TEMPLATE = """
请注意，你是一个有能力调用外部工具的智能助手。

可用工具如下:
{tools}

**请严格按照以下格式进行回应!**:

Thought: 你的思考过程，用于分析问题、拆解任务和规划下一步行动。
Action: 你决定采取的行动，只能是以下格式中的一个:
- `{{tool}}[{{params}}]`:调用一个可用工具。
- `Finish[最终答案]`:当你认为已经获得最终答案时。
当你收集到足够的信息，能够回答用户的最终问题时，你必须在Action:字段后使用 Finish[最终答案] 来输出最终答案。

现在，请开始解决以下问题:
Question: {question}
History: {history}
"""

### 工具与工具包

In [86]:
def search(query: str):
    print(f"🔍 正在进行网页搜索: {query}")
    return """截至2026年6月1日，英伟达最新发布的消费级桌面显卡是 GeForce RTX 50 系列（基于 Blackwell 架构），
    包括 RTX 5090、5080 等型号；专业/数据中心领域最新为 RTX PRO Blackwell（如 RTX PRO 6000）和 H200/H100 系
    列（H200 于2024年发布，仍属当前旗舰AI加速卡）。‌‌
    """

In [87]:
from typing import Dict, Any

class ToolExecutor:
    def __init__(self):
        self.tools: Dict[str, Dict[str, Any]] = {}

    def registerTool(self, name: str, description: str, func: callable):
        if name in self.tools:
            print(f"警告:工具 '{name}' 已存在，将被覆盖。")
        self.tools[name] = {"description": description, "func": func}
        print(f"工具 '{name}' 已注册。")

    def getTool(self, name: str) -> callable:
        return self.tools.get(name, {}).get("func")

    def getAvailableTools(self) -> str:
        return "\n".join([
            f"- {name}: {info['description']}" 
            for name, info in self.tools.items()
        ])

In [98]:
toolExecutor = ToolExecutor()
search_description = "一个网页搜索引擎。当你需要回答关于时事、事实以及在你的知识库中找不到的信息时，应使用此工具。"
toolExecutor.registerTool("Search", search_description, search)
print("\n--- 可用的工具 ---")
print(toolExecutor.getAvailableTools())

tool_name = "Search"
tool_input = "英伟达最新GPU型号"
tool_function = toolExecutor.getTool(tool_name)
observation = tool_function(tool_input)
print(observation)

工具 'Search' 已注册。

--- 可用的工具 ---
- Search: 一个网页搜索引擎。当你需要回答关于时事、事实以及在你的知识库中找不到的信息时，应使用此工具。
🔍 正在进行网页搜索: 英伟达最新GPU型号
截至2026年6月1日，英伟达最新发布的消费级桌面显卡是 GeForce RTX 50 系列（基于 Blackwell 架构），
    包括 RTX 5090、5080 等型号；专业/数据中心领域最新为 RTX PRO Blackwell（如 RTX PRO 6000）和 H200/H100 系
    列（H200 于2024年发布，仍属当前旗舰AI加速卡）。‌‌
    


### 输出解析

In [116]:
def ReAct_parse_output(self, text: str):
    
    # 从Thought: 匹配到 Action: 或文本末尾
    thought_match = re.search(r"Thought:\s*(.*?)(?=\nAction:|$)", text, re.DOTALL)
    
    # 从Action: 匹配到文本末尾
    action_match = re.search(r"Action:\s*(.*?)$", text, re.DOTALL)
    
    thought = thought_match.group(1).strip() if thought_match else None
    action = action_match.group(1).strip() if action_match else None
    return thought, action

In [117]:
text="""Thought: 首先需要获取北京今天的天气情况，之后再根据天气情况来推荐旅游景点。
Action: Get_weather[北京]

Observation: 北京当前天气:Sunny，气温26摄氏度
"""
thought, action = ReAct_parse_output(None, text)
print("thought:", thought)
print("action:", action)

thought: 首先需要获取北京今天的天气情况，之后再根据天气情况来推荐旅游景点。
action: Get_weather[北京]

Observation: 北京当前天气:Sunny，气温26摄氏度


### 工具解析

In [118]:
def ReAct_parse_action(self, action_text: str):

    # 从头匹配，先匹配字母数字下划线：get_weather，然后匹配 [，然后批匹配任意字符，最后匹配 ]
    # re.DOTALL :让 . 匹配包括换行符在内的所有字符
    match = re.match(r"(\w+)\[(.*)\]", action_text, re.DOTALL)
    if match:
        return match.group(1), match.group(2)
    return None, None

In [119]:
tool, params = ReAct_parse_action(None, action)
print("调用函数：",tool,"(",params,")")

调用函数： Get_weather ( 北京 )


### ReActAgent简单实现

In [132]:
class ReActAgent:
    def __init__(self, tool_executor: ToolExecutor, model=None, max_steps=5):
        self.model = model,
        self.tool_executor = tool_executor
        self.max_steps = max_steps
        self.history = []
        
    def run(self, question: str):
        self.history = [] 
        current_step = 0
        model, tokenizer = load_model()
        while current_step < self.max_steps:
            current_step+=1
            print(f"--- 第 {current_step} 步 ---")
            
            # 准备提示词
            available_tools = self.tool_executor.getAvailableTools()
            history_str = "\n".join(self.history)
            prompt = REACT_PROMPT_TEMPLATE.format(
                tools=available_tools,
                question=question,
                history=history_str
            )
            # print("提示词：",prompt,"\n")
            
            # 模型推理
            messages = [{"role": "user", "content": prompt}]
            think_output, output = get_qwen_output(messages, model, tokenizer)
            # print(think_output, "\n" ,output)
            # 输出解析
            thought, action = self.ReAct_parse_output(output)
            print("思考过程：",thought) if thought else None
            if not action:
                print(output)
                print(thought, action)
                break
                
            if action.startswith("Finish"):
                # 如果是Finish指令，提取最终答案并结束
                final_answer = re.match(r"Finish\[(.*)\]", action).group(1)
                print(f"🎉 最终答案: {final_answer}")
                return final_answer

            # 工具解析
            tool, params = self.ReAct_parse_action(action)
            if not tool or not params:
                print(output)
                print(tool, params)
                break

            print(f"🎬 行动: {tool}[{params}]")
            
            tool_function = self.tool_executor.getTool(tool_name)
            if not tool_function:
                observation = f"错误:未找到名为 '{tool}' 的工具。"
            else:
                observation = tool_function(params) # 调用真实工具
                
            print(f"👀 观察: {observation}")
            # 将本轮的Action和Observation添加到历史记录中
            self.history.append(f"Action: {action}")
            self.history.append(f"Observation: {observation}")

        # 循环结束
        del model
        del tokenizer
        CleanMemory()
        print("已达到最大步数，流程终止。")

In [133]:
ReActAgent.ReAct_parse_output = ReAct_parse_output
ReActAgent.ReAct_parse_action = ReAct_parse_action

In [134]:
agent = ReActAgent(toolExecutor)
agent.run("英伟达最新的GPU型号是什么？")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

--- 第 1 步 ---
思考过程： 英伟达（NVIDIA）最新推出的GPU型号是A100，这是2023年发布的，目前在市场中广泛使用。
🎬 行动: Search[NVIDIA最新GPU型号]  
Finish[NVIDIA最新GPU型号是A100]
🔍 正在进行网页搜索: NVIDIA最新GPU型号]  
Finish[NVIDIA最新GPU型号是A100
👀 观察: 截至2026年6月1日，英伟达最新发布的消费级桌面显卡是 GeForce RTX 50 系列（基于 Blackwell 架构），
    包括 RTX 5090、5080 等型号；专业/数据中心领域最新为 RTX PRO Blackwell（如 RTX PRO 6000）和 H200/H100 系
    列（H200 于2024年发布，仍属当前旗舰AI加速卡）。‌‌
    
--- 第 2 步 ---
Finish[NVIDIA最新GPU型号是A100]
None None
已达到最大步数，流程终止。


**显然0.6B的小模型还是听不懂提示词，正确结果如下**

```
问题：华为最新的手机是哪个？
--- 第 1 步 ---
Thought: 要回答这个问题，我需要查找华为最新发布的手机型号及其主要特点。这些信息可能在我的现有知识库之外，因此需要使用搜索引擎来获取最新数据。
Action: Search[华为最新手机型号及主要卖点]
🤔 思考: 要回答这个问题，我需要查找华为最新发布的手机型号及其主要特点。这些信息可能在我的现有知识库之外，因此需要使用搜索引擎来获取最新数据。
🎬 行动: Search[华为最新手机型号及主要卖点]
🔍 正在执行 [SerpApi] 网页搜索: 华为最新手机型号及主要卖点
👀 观察: [1] 华为手机- 华为官网
智能手机 ; Mate 系列. 非凡旗舰 · HUAWEI Mate XTs. 非凡大师 ; Pura 系列. 先锋影像 · HUAWEI Pura 80 Pro+ ; Pocket 系列. 美学新篇. HUAWEI Pocket 2 ; nova 系列. 专业人像.

[2] 2025年华为手机哪一款性价比高？华为手机推荐与市场分析 ...
现在华为手机最大的卖点只剩下鸿蒙HarmonyOS系统，以及饱受争议的品牌信仰。 这里推荐目前值得入手的几款华为系列手机，根据不同预算自行选择:. 华为目前最受欢迎，也是搭载 ...

[3] 2025年华为新款手机哪个性价比高？10款华为新款手机推荐
选华为主要还是要推荐高端手机，Mate 70和Pura 70系列是最新发布的旗舰机型。 HUAWEI Mate 70. 优点是，拍照配置依旧顶级，全焦段覆盖，适合专业摄影，做工出色，户外抗摔 ...

--- 第 2 步 ---
Thought: 根据搜索结果，华为最新发布的旗舰机型包括Mate 70和Pura 80 Pro+。为了确定最新型号及其主要卖点，我将重点放在这些信息上。从提供的链接来看，Mate 70系列和Pura 80 Pro+都是近期发布的产品，但具体哪一个是“最新”还需要进一步确认。同时，我可以从这些信息中提取出它们的主要
卖点。
Action: Finish[根据最新信息，华为的最新手机可能是HUAWEI Pura 80 Pro+或HUAWEI Mate 70。其中，HUAWEI Mate 70的主要卖点包括顶级的拍照配置，全焦段覆盖，适合专业摄影，做工出色，并且具有良好的户外抗摔性能。而HUAWEI Pura 80 Pro+则强调了先锋影像技术。]
🤔 思考: 根据搜索结果，华为最新发布的旗舰机型包括Mate 70和Pura 80 Pro+。为了确定最新型号及其主要卖点，我将重点放在这些信息上。从提供的链接来看，Mate 70系列和Pura 80 Pro+都是近期发布的产品，但具体哪一个是“最新”还需要进一步确认。同时，我可以从这些信息中提取出它们的主要 
卖点。
🎉 最终答案: 根据最新信息，华为的最新手机可能是HUAWEI Pura 80 Pro+或HUAWEI Mate 70。其中，HUAWEI Mate 70的主要卖点包括顶级的拍照配置，全焦段覆盖，适合专业摄影，做工出色，并且具有良好的户外抗摔性能。而HUAWEI Pura 80 Pro+则强调了先锋影像技术。